# Khám phá dữ liệu (Data Exploration)
Notebook này bám theo logic trong `source/data/load_dataset.py` và `source/data/filter_labor.py` nhưng viết trực tiếp trong notebook (không import file nội bộ).
Để tránh treo máy, mặc định dùng `DATA_SPLIT = "data[:2000]"` (có thể đổi thành `"data"` để chạy toàn bộ).

In [1]:
from datasets import get_dataset_config_names, load_dataset

In [2]:
dataset_name = "th1nhng0/vietnamese-legal-documents"
DATA_SPLIT = "data[:2000]"  # Doi thanh "data" neu muon chay toan bo

configs = get_dataset_config_names(dataset_name)
print("1. Cac config cua dataset:", configs)
print("Dang su dung split:", DATA_SPLIT)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.51k [00:00<?, ?B/s]

1. Cac config cua dataset: ['metadata', 'relationships', 'content', 'legacy']
Dang su dung split: data[:2000]


In [3]:
metadata_ds = load_dataset(dataset_name, "metadata", split=DATA_SPLIT)
content_ds = load_dataset(dataset_name, "content", split=DATA_SPLIT)

print("2. Cac cot cua metadata:", metadata_ds.column_names)
print("3. Cac cot cua content:", content_ds.column_names)

# Chuyen sang list dict de xu ly nhanh (da gioi han so mau o tren)
metadata_rows = [dict(row) for row in metadata_ds]
content_rows = [dict(row) for row in content_ds]

data/metadata.parquet:   0%|          | 0.00/14.1M [00:00<?, ?B/s]

Generating data split: 0 examples [00:00, ? examples/s]

data/content.parquet:   0%|          | 0.00/412M [00:00<?, ?B/s]

Generating data split: 0 examples [00:00, ? examples/s]

2. Cac cot cua metadata: ['id', 'title', 'so_ky_hieu', 'ngay_ban_hanh', 'loai_van_ban', 'ngay_co_hieu_luc', 'ngay_het_hieu_luc', 'nguon_thu_thap', 'ngay_dang_cong_bao', 'nganh', 'linh_vuc', 'co_quan_ban_hanh', 'chuc_danh', 'nguoi_ky', 'pham_vi', 'thong_tin_ap_dung', 'tinh_trang_hieu_luc']
3. Cac cot cua content: ['id', 'content_html']


In [4]:
# 4. Merge metadata va content bang khoa 'id' (theo logic trong load_dataset.py)
join_key = "id"
content_map = {str(row.get(join_key)): row.get("content_html", "") for row in content_rows}

merged_docs = []
for meta in metadata_rows:
    m_id = str(meta.get(join_key))
    if m_id in content_map:
        merged_docs.append({**meta, "id": m_id, "content_html": content_map[m_id]})

print("4. Join metadata-content bang field:", join_key)
print(f"Tong so van ban sau khi merge: {len(merged_docs)}")

4. Join metadata-content bang field: id
Tong so van ban sau khi merge: 1873


In [5]:
# Danh sach tu khoa chuyen nganh (dua theo filter_labor.py)
labor_keywords = [
    # lao dong chung
    "lao động",
    "bộ luật lao động",
    "quan hệ lao động",
    "việc làm",
    "học nghề",
    "đào tạo nghề",

    # hợp đồng
    "hợp đồng lao động",
    "giao kết hợp đồng",
    "chấm dứt hợp đồng",
    "đơn phương chấm dứt",
    "thử việc",

    # chủ thể
    "người lao động",
    "người sử dụng lao động",

    # lương và thu nhập
    "tiền lương",
    "tiền công",
    "lương tối thiểu",
    "thưởng",

    # thời gian làm việc
    "thời giờ làm việc",
    "thời giờ nghỉ ngơi",
    "làm thêm giờ",
    "nghỉ hằng năm",
    "nghỉ lễ",

    # bảo hiểm
    "bảo hiểm xã hội",
    "bảo hiểm thất nghiệp",
    "bảo hiểm y tế",
    "trợ cấp thôi việc",
    "trợ cấp mất việc",

    # nghỉ việc
    "nghỉ việc",
    "sa thải",
    "kỷ luật lao động",

    # an toàn lao động
    "an toàn vệ sinh lao động",
    "tai nạn lao động",
    "bệnh nghề nghiệp",

    # tổ chức lao động
    "công đoàn",
    "thỏa ước lao động tập thể",
    "đình công",
    "tranh chấp lao động",

    # đối tượng đặc biệt
    "lao động nữ",
    "thai sản",
    "người chưa thành niên",
    "lao động nước ngoài",
    "giấy phép lao động"
]

def is_labor_related(doc):
    text_to_search = f"{doc.get('title', '')} {doc.get('content_html', '')}".lower()
    return any(keyword in text_to_search for keyword in labor_keywords)

labor_docs = [doc for doc in merged_docs if is_labor_related(doc)]
print(f"5. So luong van ban lien quan den lao dong: {len(labor_docs)}")

5. So luong van ban lien quan den lao dong: 738


In [6]:
print("--- 5 SAMPLE DOCUMENTS ---")
for idx, doc in enumerate(labor_docs[:5]):
    title = doc.get("title", "")
    doc_type = doc.get("loai_van_ban") or doc.get("doc_type", "")
    effective_date = doc.get("ngay_co_hieu_luc") or doc.get("effective_date", "")
    text_preview = str(doc.get("content_html", ""))[:250]

    print(f"Sample #{idx + 1}")
    print(f"Title:          {title}")
    print(f"Type:           {doc_type}")
    print(f"Effective Date: {effective_date}")
    print(f"Text preview:   {text_preview}...\n")
    print("-" * 50)

--- 5 SAMPLE DOCUMENTS ---
Sample #1
Title:          Về viẹc quản lý và quy định thống nhất phí chợ trên địa bàn tỉnh Lâm Đồng
Type:           Quyết định
Effective Date: 19/11/1999
Text preview:   <table class="detailcontent" width="100%" border="0" id="content">
                <tr>
                    <td colspan="3">
                        <div align="justify">
                            <p>
	
</p>
<p>
	
</p>
<p>
	</p><title></title>
...

--------------------------------------------------
Sample #2
Title:          Về việc cho phép thành lập Quỹ bảo trợ nạn nhân chất độc da cam tỉnh Lâm Đồng và ban hành Quy chế tổ chức hoạt động của quỹ
Type:           Quyết định
Effective Date: 03/06/1999
Text preview:   <table class="detailcontent" width="100%" border="0" id="content">
                <tr>
                    <td colspan="3">
                        <div align="justify">
                            <p>
	
</p>
<p>
	
</p>
<p>
	</p><title></title>
...

------------------------------